In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import os

# Drive  zip file  path
zip_path = '/content/drive/MyDrive/train.zip'
# Extraction the  folder
extract_path = 'content/dataset'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction is completed!")
else:
    print("File is not found")

In [ ]:
# check the Folder structure  
base_dir = 'content/dataset'
folders = os.listdir(base_dir)
print(f"Folders found: {folders}")

In [ ]:
import glob

def get_data_paths(base_path):
    #  if the folders  name 'cat' aur 'dog' 
   cats = glob.glob(os.path.join(base_path, 'cat/*.jpg'))
   dogs = glob.glob(os.path.join(base_path, 'dog/*.jpg'))

    #lable name for cat is 0 and dogs 1
    paths = cats + dogs
    labels = [0] * len(cats) + [1] * len(dogs)

    return paths, labels

# Extraction path to take the data
image_paths, labels = get_data_paths('/content/dataset')
print(f"Total images: {len(image_paths)}")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch

# check the GPU Is Available or not 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"we are  '{device}' Use.")

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Convolutional layers to extract features
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

        # Fully connected layers classification
        self.fc1 = nn.Linear(32 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, 2) # 2 outputs: Cat ya Dog

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 56 * 56) # Flattening the array
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Model create instance for GPU
model = SimpleCNN().to(device)
print("Model tayyar hai!")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
class PetDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        # Read Images
        img = cv2.imread(self.paths[idx])
        if img is None:
            # is any image is currpt then skip it
            return torch.zeros((3, 224, 224)), torch.tensor(self.labels[idx])

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Resize image to 224x224
        img = cv2.resize(img, (224, 224))

        # NumPy array to normalize and convert to tensor
        img = img.transpose((2, 0, 1))
        img_tensor = torch.from_numpy(img).float() / 255.0

        label = torch.tensor(self.labels[idx])
        return img_tensor, label

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader

# 1. first create dataset object
dataset = PetDataset(image_paths,labels)

# 2. define DataLoader
# batch_size=32  mean 32 image is processing in sametime
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

print("train_loader tayyar hai!")

# Loss function and Optimizer define
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
def train_model(epochs):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            # Data is loading on  GPU
            images, labels = images.to(device), labels.to(device)

            # Gradients is zero
            optimizer.zero_grad()

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

# Training start karein (e.g 10 epochs)
train_model(10)

In [ ]:
import matplotlib.pyplot as plt
import random

# Model  evaluation mode  set
model.eval()

# to take the random image from Dataset
idx = random.randint(0, len(image_paths)-1)
test_path = image_paths[idx]
true_label = "Dog" if labels[idx] == 1 else "Cat"

# Image the process
img = cv2.imread(test_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (224, 224))
img_tensor = torch.from_numpy(img_resized.transpose((2, 0, 1))).float().unsqueeze(0) / 255.0
img_tensor = img_tensor.to(device)

# Prediction
with torch.no_grad():
    output = model(img_tensor)
    _, predicted = torch.max(output, 1)
    pred_label = "Dog" if predicted.item() == 1 else "Cat"

# show the result
plt.imshow(img_rgb)
plt.title(f"Original: {true_label} | Model Ask: {pred_label}")
plt.axis('off')
plt.show()

if true_label == pred_label:
    print("Congragulation. ")
else:
    print("Oh! Misconception.")